In [ ]:
# Import necessary libraries
import torch
import numpy as np
import xgboost as xgb
from PIL import Image
import torchvision.transforms as transforms
import os
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
import cv2

SEED = 123
def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)


In [ ]:
# Paths and Parameters
checkpoint_path = os.path.join("..", "..", "files", "alexnet.pth")
xgb_model_path = os.path.join("..", "..", "files", "final_xgboost_model.json")

INPUT_SIZE = (227,227)
MEAN = (0.5960, 0.4489, 0.4046)
STD = (0.2573, 0.2147, 0.2100)
BATCH_SIZE = 64
# NUM_WORKERS=0: multiprocessing DataLoader workers proved unreliable inside
# this Jupyter/VS Code kernel on Windows (crashed with "exited unexpectedly"
# even after fixing pickling and cv2 thread contention). 0 is single-process
# but reliable.
NUM_WORKERS = 0

main_data_dir = os.path.join("..", "..", "..", "Dataset")
val_dir = os.path.join(main_data_dir, "val")
test_dir = os.path.join(main_data_dir, "test")

val_dataset = datasets.ImageFolder(root=val_dir)
class_names = list(val_dataset.classes)
print("Class to label mapping:", val_dataset.class_to_idx)

In [ ]:
import sys
# Transform classes live in models/preprocessing.py (not defined inline here)
# so that Windows DataLoader worker subprocesses (which use 'spawn') can
# actually pickle/import them. Classes defined in a notebook cell resolve to
# the ipykernel launcher's __main__ in a spawned worker, not this notebook's
# namespace, so num_workers>0 would otherwise hang or fail silently.
sys.path.insert(0, os.path.abspath(os.path.join("..", "..")))
from preprocessing import CLAHETransform

In [ ]:
transform_val_test = transforms.Compose([
    transforms.Resize(INPUT_SIZE),
    CLAHETransform(clip_limit=2.0, tile_grid_size=(8, 8)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

val_dataset = datasets.ImageFolder(root=val_dir, transform=transform_val_test)
test_dataset = datasets.ImageFolder(root=test_dir, transform=transform_val_test)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)


In [ ]:
import torch
import torch.nn as nn
from torchvision import models

model = models.alexnet(weights=None)  

model.classifier = nn.Sequential(
    *list(model.classifier.children())[:3]  
)

state_dict = torch.load(checkpoint_path, weights_only=True)

new_state_dict = {}
for k, v in state_dict.items():
    new_key = k.replace("model.", "") if k.startswith("model.") else k
    if new_key in model.state_dict().keys():
        new_state_dict[new_key] = v

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

model.load_state_dict(new_state_dict, strict=False)  

for param in model.parameters():
    param.requires_grad = False


model.eval()
print(f"Loaded model from {checkpoint_path}")
print(model)

# from torchsummary import summary
# input_size = (3, 227, 227)
# summary(model, input_size=input_size, device=str(device))


In [ ]:
xgb_model = xgb.XGBClassifier()
xgb_model.load_model(xgb_model_path)
print(f"Loaded XGBoost model from {xgb_model_path}")

In [ ]:
def evaluate_model(data_loader, set_name, export_txt=False):
    true_labels = []
    predicted_labels = []
    image_paths = []  # To store image file paths
    confidences = []  # To store confidence scores

    # Access the dataset from the DataLoader to get image file paths
    dataset = data_loader.dataset
    sample_idx = 0  # running offset into dataset.samples, robust to any batch size

    for images, labels in tqdm(data_loader, desc=f"Evaluating {set_name}"):
        images = images.to(device)
        labels = labels.cpu().numpy()

        with torch.no_grad():
            features = model(images).cpu().numpy()
        probabilities = xgb_model.predict_proba(features)  # Get probabilities for each class
        predictions = np.argmax(probabilities, axis=1)  # Predicted labels
        confidence_scores = np.max(probabilities, axis=1) * 100   # Confidence scores for predicted labels

        true_labels.extend(labels)
        predicted_labels.extend(predictions)
        confidences.extend(confidence_scores)

        # Collect image file paths from the dataset (loader has shuffle=False,
        # so sample order matches dataset.samples order)
        batch_size = len(labels)
        image_paths.extend(dataset.samples[i][0] for i in range(sample_idx, sample_idx + batch_size))
        sample_idx += batch_size

    # Calculate and print accuracy
    accuracy = accuracy_score(true_labels, predicted_labels)
    print(f"\n{set_name} Accuracy: {accuracy * 100:.2f}%")

    # Print classification report
    print(f"\n{set_name} Classification Report:\n")
    print(classification_report(true_labels, predicted_labels, target_names=class_names))

    # Confusion matrix
    cm = confusion_matrix(true_labels, predicted_labels)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted Labels')
    plt.ylabel('True Labels')
    plt.title(f'{set_name} Confusion Matrix on AlexNet With XGBoost')
    plt.show()

    if export_txt:
        output_file = f"{set_name}_with_xgb_predictions.txt"
        with open(output_file, "w") as file:
            file.write("Image File, True Label, Predicted Label, Confidence\n")
            for img_path, true_label, pred_label, confidence in zip(image_paths, true_labels, predicted_labels, confidences):
                true_class_name = class_names[true_label]
                pred_class_name = class_names[pred_label]
                file.write(f"{img_path}, {true_class_name}, {pred_class_name}, {confidence:.2f}%\n")
        print(f"Predictions saved to {output_file}")


In [ ]:

evaluate_model(val_loader, "Validation Set")

In [ ]:
evaluate_model(test_loader, "Test Set", export_txt=True)